In [1]:
import tensorflow as tf

In [2]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [5]:
tf.test.is_gpu_available

<function tensorflow.python.framework.test_util.is_gpu_available(cuda_only=False, min_cuda_compute_capability=None)>

In [6]:
# Check if GPU is available
if tf.config.list_physical_devices('GPU'):
    print("GPU is available and being used!")
    print(tf.config.list_physical_devices('GPU'))
else:
    print("GPU is not available, using CPU instead.")

GPU is available and being used!
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:

!pip install librosa


  Using cached librosa-0.10.2.post1-py3-none-any.whl.metadata (8.6 kB)
  Using cached audioread-3.0.1-py3-none-any.whl.metadata (8.4 kB)
  Using cached scipy-1.14.1-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached scikit_learn-1.6.0-cp310-cp310-win_amd64.whl.metadata (15 kB)
  Using cached joblib-1.4.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached numba-0.60.0-cp310-cp310-win_amd64.whl.metadata (2.8 kB)
  Using cached soundfile-0.12.1-py2.py3-none-win_amd64.whl.metadata (14 kB)
  Using cached pooch-1.8.2-py3-none-any.whl.metadata (10 kB)
  Using cached lazy_loader-0.4-py3-none-any.whl.metadata (7.6 kB)
  Using cached msgpack-1.1.0-cp310-cp310-win_amd64.whl.metadata (8.6 kB)
  Using cached llvmlite-0.43.0-cp310-cp310-win_amd64.whl.metadata (4.9 kB)
  Using cached cffi-1.17.1-cp310-cp310-win_amd64.whl.metadata (1.6 kB)
  Using cached pycparser-2.22-py3-none-any.whl.metadata (943 bytes)
Using cached librosa-0.10.2.post1-py3-none-any.whl (260 kB)
Using cached audioread-3.0.1-p

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\Hariz\\anaconda3\\envs\\py310\\Lib\\site-packages\\scipy\\special\\_precompute\\lambertw.py'
Consider using the `--user` option or check the permissions.



In [1]:
import glob
import os
import librosa
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler

# Path to the metadata file
metadata_file_path = r'D:\download\Compressed\DF-keys-full\keys\DF\CM\trial_metadata.txt'

# Dictionary to hold file identifiers and labels
labels = {}
with open(metadata_file_path, 'r') as f:
    for line in f:
        parts = line.strip().split()
        file_identifier = parts[1]
        label = parts[5]
        labels[file_identifier] = label

# Directory containing the undersampled audio files
undersampled_audio_dir = r'D:\audio_files\undersampled_audio'
audio_files = glob.glob(os.path.join(undersampled_audio_dir, '*.flac'))

valid_audio_files = []
valid_labels = []

for audio_file in audio_files:
    file_name = os.path.basename(audio_file)
    file_identifier = file_name.split('.')[0]
    if file_identifier in labels:
        valid_audio_files.append(audio_file)
        valid_labels.append(labels[file_identifier])

print(f"Found {len(valid_audio_files)} audio files with corresponding labels.")

# Audio to MFCC conversion function
def audio_to_mfcc(audio_file, sr=22050, n_mfcc=13):
    audio, sample_rate = librosa.load(audio_file, sr=sr)
    mfcc = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=n_mfcc)
    return mfcc

def pad_or_truncate(mfcc, max_time_steps=400):
    if mfcc.shape[1] > max_time_steps:
        return mfcc[:, :max_time_steps]
    elif mfcc.shape[1] < max_time_steps:
        padding = max_time_steps - mfcc.shape[1]
        return np.pad(mfcc, ((0, 0), (0, padding)), mode='constant')
    return mfcc

# Preprocessing function with normalization
def preprocess_audio_files_rnn(audio_files, labels, n_mfcc=13, max_time_steps=400):
    mfcc_data = []
    label_data = []
    scaler = StandardScaler()
    
    for audio_file, label in zip(audio_files, labels):
        mfcc = audio_to_mfcc(audio_file, n_mfcc=n_mfcc)
        mfcc = pad_or_truncate(mfcc, max_time_steps=max_time_steps).T
        mfcc = scaler.fit_transform(mfcc)
        mfcc_data.append(mfcc)
        label_data.append(1 if label == 'spoof' else 0)

    return np.array(mfcc_data), np.array(label_data)

X, y = preprocess_audio_files_rnn(valid_audio_files, valid_labels)
print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

def stratified_split_70_30(X, y, test_size=0.2, random_state=42):
    strat_split = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    for train_idx, test_idx in strat_split.split(X, y):
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = stratified_split_70_30(X, y)
print(f"Training set distribution: {np.bincount(y_train)}")
print(f"Test set distribution: {np.bincount(y_test)}")

def build_rnn_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        LSTM(128, return_sequences=True, activation='tanh'),
        Dropout(0.3),
        BatchNormalization(),
        LSTM(64, return_sequences=True, activation='tanh'),
        Dropout(0.3),
        BatchNormalization(),
        LSTM(32, activation='tanh'),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    return model

input_shape = (X_train.shape[1], X_train.shape[2])
rnn_model = build_rnn_model(input_shape)
rnn_model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = rnn_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

test_loss, test_accuracy = rnn_model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

y_pred = (rnn_model.predict(X_test) > 0.5).astype(int).flatten()
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

conf_matrix = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(conf_matrix)


Found 45234 audio files with corresponding labels.
Shape of X: (45234, 400, 13)
Shape of y: (45234,)
Training set distribution: [18093 18094]
Test set distribution: [4524 4523]
Epoch 1/50
453/453 [==============================] - 75s 132ms/step - loss: 0.5608 - accuracy: 0.7293 - val_loss: 0.4974 - val_accuracy: 0.7666
Epoch 2/50
453/453 [==============================] - 62s 136ms/step - loss: 0.5055 - accuracy: 0.7668 - val_loss: 0.4621 - val_accuracy: 0.7860
Epoch 3/50
453/453 [==============================] - 64s 142ms/step - loss: 0.4866 - accuracy: 0.7756 - val_loss: 0.4572 - val_accuracy: 0.7903
Epoch 4/50
453/453 [==============================] - 71s 156ms/step - loss: 0.4736 - accuracy: 0.7819 - val_loss: 0.4403 - val_accuracy: 0.7991
Epoch 5/50
453/453 [==============================] - 68s 149ms/step - loss: 0.4604 - accuracy: 0.7875 - val_loss: 0.4319 - val_accuracy: 0.7980
Epoch 6/50
453/453 [==============================] - 67s 149ms/step - loss: 0.4483 - accuracy: 0.